# Compute Weighted F1 Scores from F1 Outputs

In [ ]:
import json
import os
import numpy as np
from pathlib import Path

CLAIMS_PATH = '/content/drive/MyDrive/infra_fm/claims.json'
RESULTS_ROOT = '/content/drive/MyDrive/infra_fm/results'
WEIGHTED_OUT = '/content/drive/MyDrive/infra_fm/weighted_f1_computed.json'
TOLERANCE = 0.0005

with open(CLAIMS_PATH) as f:
    claims = json.load(f)

# Remap AlphaEarth as before
for table_name, table_data in list(claims.items()):
    if 'alphaearth_v2_spatial' in table_data:
        table_data['alphaearth_v2'] = table_data.pop('alphaearth_v2_spatial')


# =========================================================================
# STEP 1: Compute weighted F1 from per-seed confusion matrices
# =========================================================================
print('=' * 80)
print('STEP 1: Computing weighted F1 from per-seed confusion matrices')
print('=' * 80)

weighted_by_exp = {}

for subdir in sorted(os.listdir(RESULTS_ROOT)):
    sub_path = os.path.join(RESULTS_ROOT, subdir)
    if not os.path.isdir(sub_path): continue

    # Find per-seed result JSONs (pattern: <prefix>_seed<N>_results.json)
    seed_files = sorted([f for f in os.listdir(sub_path)
                         if '_seed' in f and f.endswith('_results.json')])
    if not seed_files:
        continue

    exp_key = subdir.replace('fm_eval_', '')

    per_seed_wf1 = []
    for fname in seed_files:
        with open(os.path.join(sub_path, fname)) as fp:
            payload = json.load(fp)

        # Per-seed JSONs are typically wrapped: {'linear_probe': {...}} or
        # {'finetune': {...}}. Unwrap.
        test = None
        for wrapper in ('linear_probe', 'finetune', 'supervised', 'random_features'):
            if wrapper in payload and isinstance(payload[wrapper], dict):
                inner = payload[wrapper]
                if 'test' in inner:
                    test = inner['test']
                    break
        if test is None and 'test' in payload:
            test = payload['test']
        if test is None:
            continue

        cm = np.array(test.get('confusion', []))
        f1s = np.array(test.get('per_class_f1', []))
        if cm.size == 0 or f1s.size == 0 or cm.shape[0] != f1s.shape[0]:
            continue

        support = cm.sum(axis=1)                # true-label counts per class
        total = support.sum()
        if total > 0:
            wf1 = float(np.sum(support * f1s) / total)
            per_seed_wf1.append(wf1)

    if per_seed_wf1:
        arr = np.array(per_seed_wf1)
        weighted_by_exp[exp_key] = {
            'mean':     float(arr.mean()),
            'std':      float(arr.std(ddof=0)),
            'per_seed': [float(v) for v in per_seed_wf1],
            'n_seeds':  len(per_seed_wf1),
        }

# Save durable source
with open(WEIGHTED_OUT, 'w') as f:
    json.dump(weighted_by_exp, f, indent=2)

print(f'\nComputed weighted F1 for {len(weighted_by_exp)} experiments '
      f'(saved to {WEIGHTED_OUT}):')
for exp_key in sorted(weighted_by_exp.keys()):
    v = weighted_by_exp[exp_key]
    print(f'  {exp_key:<40s}  {v["mean"]:.3f} ± {v["std"]:.3f}  (n_seeds={v["n_seeds"]})')
